In [130]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load data
df = pd.read_excel("TAWE_Factors_Data_Reviewed.xlsx")
all_quotes = df['Topic Quote'].dropna().tolist()
print(f"Number of quotes to embed: {len(all_quotes)}")

# Load local model
model_path = "/home/ec2-user/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(model_path, local_files_only=True)
print("Embedding model loaded!")


Number of quotes to embed: 212
Embedding model loaded!


In [131]:
import numpy as np
batch_size = 50
embeddings_list = []

for i in range(0, len(all_quotes), batch_size):
    batch = all_quotes[i:i+batch_size]
    batch_embeds = embed_model.encode(batch, show_progress_bar=False)
    embeddings_list.extend(batch_embeds)

embeddings_array = np.array(embeddings_list)
print("Embeddings shape:", embeddings_array.shape)




Embeddings shape: (212, 384)


In [132]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------
# Download NLTK resources
nltk.download("stopwords")
nltk.download("punkt")

# Load Excel file
df = pd.read_excel("Underwriter Trust Score.xlsx")

# Define filler words
filler_words = {
    "yeah", "you know", "uh", "um", "like", "so", "actually", "think", "ai", "thats", "would", "know", "im", "definitely", "could",
    "absolutelty","basically", "right", "ok", "okay", "well", "hmm","say","going"
}

# Get NLTK stopwords
stop_words = set(stopwords.words("english"))

# Combine stop words and filler words
combined_stop_words = stop_words.union(filler_words)
combined_stop_words_list = list(combined_stop_words)  # Must be a list for TfidfVectorizer

# Define the label columns (manual one-hot columns, adjust index as needed)
labels = df.columns[4:10]  # assuming columns 4:10 are the manual labels

# For each label, extract the quotes that are manually labeled as 1
keywords_by_label = {}

for label in labels:
    print(f"\nProcessing label: {label}")
    
    # Filter quotes for this label
    quotes = df[df[label] == 1]['Topic Quote'].dropna().tolist()
    
    if len(quotes) == 0:
        print(f"No quotes found for label: {label}")
        keywords_by_label[label] = []
        continue
    
    # Initialize TF-IDF vectorizer
    vectorizer = TfidfVectorizer(stop_words=combined_stop_words_list, ngram_range=(1,2), max_features=50)
    tfidf_matrix = vectorizer.fit_transform(quotes)
    
    # Get feature names and their mean TF-IDF score
    feature_names = vectorizer.get_feature_names_out()
    tfidf_scores = tfidf_matrix.mean(axis=0).A1  # Average across all quotes
    tfidf_dict = dict(zip(feature_names, tfidf_scores))
    
    # Sort features by score
    sorted_features = sorted(tfidf_dict.items(), key=lambda x: x[1], reverse=True)
    keywords_by_label[label] = [word for word, score in sorted_features[:20]]  # Top 20 keywords
    
    print(f"Top keywords for {label}: {keywords_by_label[label]}")

# keywords_by_label now contains a dictionary of label -> list of keywords



Processing label: Adaptability
Top keywords for Adaptability: ['need', 'human', 'people', 'decision', 'different', 'loan', 'making', 'good', 'text', 'credit', 'something', 'approved', 'number', 'feel', 'make', 'scenarios', 'explanation', 'still', 'application', 'decline']

Processing label: Intent to use
Top keywords for Intent to use: ['need', 'look', 'useful', 'us', 'human', 'concerns', 'explanation', 'helps', 'approved', 'manual', 'credit', 'making', 'help', 'use', 'decision', 'yes', 'find', 'get', 'make', 'decline']

Processing label: System Performance
Top keywords for System Performance: ['data', 'system', 'credit', 'kind', 'get', 'loan', 'important', 'course', 'security', 'compliance', 'human', 'regulatory', 'trust', 'thing', 'take', 'approved', 'lot', 'one', 'different', 'particular']

Processing label: System Understanding
Top keywords for System Understanding: ['need', 'system', 'data', 'explanation', 'important', 'underwriting', 'want', 'decision', 'make', 'credit', 'use', 

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/ec2-user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [133]:
# Step 2: Define keyword dictionary (expanded)
keyword_dict = {
    'Adaptability': [
        'different', 'scenario', 'options', 'exceptions', 'multiple ways', 'flexible',
        'case by case', 'individual situations', 'customized', 'threshold', 'various',
        'alternative', 'variation','certain','specific','everybody','different','human element','opinion based',
        'decision based','humans','context'
    ],
    'Intent to use': [
        'would use', 'intend', 'envisage', 'consider', 'trust',
        'approval', 'find useful', 'planning to use', 'adopt', 'apply','help','need'
    ],
    'System Performance': [
        'efficient', 'fast', 'time-saving', 'reduce work', 'assist', 'benefit', 'streamline',
        'save time', 'accurate', 'speed', 'reliable', 'performance', 'throughput', 'processing',
        'capability', 'workflow', 'automation','regulatory','data','security','compliance','model'
    ],
    'System Understanding': [
        'understand', 'explain', 'explanation', 'clarity', 'transparency', 'interpretable',
        'reasoning', 'insight', 'visualize', 'graphs', 'text', 'analysis', 'model understanding',
        'comprehend', 'interpret'
    ],
    'Openness to Experimentation': [
        'try', 'experiment', 'explore', 'open', 'testing', 'play', 'prototype', 'learning',
        'trial', 'iterate', 'tinker', 'test cases', 'adjust', 'pilot','future','all for it'
    ],
    'Percieved Usefulness': [
        'help', 'assist', 'benefit', 'value', 'productive', 'decision', 'support', 'rationale',
        'justify', 'advantage', 'useful', 'insight', 'impact', 'effectiveness', 'reasoning','useful'
    ]
}


In [134]:
# Step 0: Imports
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from nltk.stem import WordNetLemmatizer

# Step 1: Load your Excel file
df = pd.read_excel("Underwriter Trust Score.xlsx")
quotes = df['Topic Quote'].dropna().tolist()

# Step 2: Define keyword dictionary (expanded)
keyword_dict = {
    'Adaptability': [
        # From definition: flexible, scenario-aware, personalized, contextual
        'flexible', 'adapt', 'adaptable', 'adjust', 'personalized', 'tailor',
        'customized', 'case by case', 'individual', 'context', 'contextual',
        'exceptions', 'edge case', 'variation', 'scenario', 'scenarios',
        'multiple outcomes', 'options', 'different user', 'cognitive style',
        'preferences', 'various formats', 'graphs', 'text', 'nuanced',
        'human judgment', 'human','different'
    ],

    'Intent to use': [
        # From definition: willingness, likelihood, conditional use, adopt
        'would use', 'intend', 'intention', 'likely to use', 'plan to use',
        'consider using', 'willing to adopt', 'adopt', 'use it', 'apply it',
        'would apply', 'trust enough to use', 'conditional use', 'would try',
        'need before using', 'approval', 'would rely', 'use regularly'
    ],

    'System Performance': [
        # From definition: accuracy, reliability, speed, efficiency, regulatory abilities
        'accurate', 'accuracy', 'reliable', 'reliability', 'fast', 'speed',
        'efficient', 'efficiency', 'performance', 'timely', 'processing',
        'handle complex', 'underwriting', 'large datasets', 'risk assessment',
        'fraud detection', 'regulatory', 'compliance', 'secure', 'security',
        'validate', 'validation', 'error', 'low error', 'model performance'
    ],

    'System Understanding': [
        # From definition: understanding logic, rationale, explainability, transparency
        'understand', 'explain', 'explanation', 'why it decided', 'logic',
        'algorithm', 'decision-making', 'rationale', 'reasoning', 'transparent',
        'transparency', 'clarity', 'interpretable', 'interpretability',
        'model logic', 'data inputs', 'visual explanation', 'visualizations',
        'graphs', 'charts', 'how it works', 'insight', 'model reasoning'
    ],

    'Openness to Experimentation': [
        # From definition: trying, exploring, testing, piloting
        'try', 'trying', 'experiment', 'experimenting', 'explore', 'exploring',
        'testing', 'test cases', 'pilot', 'piloting', 'prototype', 'iterate',
        'iterative', 'learning', 'play with', 'tinker', 'trial', 'scenario testing',
        'open to trying', 'willing to explore', 'test scenarios','all for it'
    ],

    'Percieved Usefulness': [
        # From definition: improves workflow, efficiency, support decision-making
        'useful', 'usefulness', 'helpful', 'help', 'assist', 'support',
        'benefit', 'valuable', 'value', 'improve efficiency', 'reduce workload',
        'save time', 'productivity', 'better decisions', 'decision quality',
        'actionable insights', 'insightful', 'streamline', 'practical benefit',
        'improve workflow', 'add value', 'enhance work','use'
    ]
}



# Step 3: Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join([lemmatizer.lemmatize(w) for w in text.lower().split()])

# Step 4: Keyword-based classification
def classify_by_keywords(quote, keyword_dict):
    quote_lemmatized = lemmatize_text(quote)
    labels = []
    for label, keywords in keyword_dict.items():
        keywords_lemmatized = [lemmatizer.lemmatize(k) for k in keywords]
        if any(k in quote_lemmatized for k in keywords_lemmatized):
            labels.append(label)
    return labels

df['Keyword Labels'] = df['Topic Quote'].apply(lambda x: classify_by_keywords(x, keyword_dict))

# Step 5: Load embedding model (MiniLM)
model_path = "/home/ec2-user/all-MiniLM-L6-v2"  # adjust if needed
embed_model = SentenceTransformer(model_path, local_files_only=True)
print("Embedding model loaded!")

# Step 6: Create label example sentences (one or more representative sentences per label)
label_examples = {
    'Adaptability': [
        "something that allows it to, or the people using to have options",
        "AI for credit assessment - it needs a person"
    ],
    'Intent to Use': [
        " that basically helps explain",
        "Oh, most definitely. It'll it'll streamline everything."
    ],
    'System Performance': [
        "The system should be efficient, fast and save time.",
        "Reliable performance and processing speed are critical."
    ],
    'System Understanding': [
        "I need to understand and interpret the system outputs.",
        "Explanations and insights make the model more transparent."
    ],
    'Openness to Experimentation': [
        "I am willing to try and experiment with the system.",
        "Testing, piloting, and iterative learning are important."
    ],
    'Percieved Usefulness': [
        "This system is helpful and adds value to our work.",
        "It assists in making better decisions and improves productivity."
    ]
}

# Step 7: Compute embeddings for label examples
label_embeddings = {}
for label, examples in label_examples.items():
    label_embeddings[label] = embed_model.encode(examples, convert_to_tensor=True)

# Step 8: Hybrid classification function
def hybrid_classify(quote, keyword_dict, label_embeddings, embed_model, threshold=0.65):
    # Start with keyword labels
    labels = classify_by_keywords(quote, keyword_dict)
    
    # Embedding-based similarity
    quote_emb = embed_model.encode(quote, convert_to_tensor=True)
    for label, examples_emb in label_embeddings.items():
        sim = util.cos_sim(quote_emb, examples_emb)  # shape (1, n_examples)
        max_sim = sim.max().item()
        if max_sim >= threshold and label not in labels:
            labels.append(label)
    return labels

# Step 9: Apply hybrid classification
df['Predicted Labels'] = df['Topic Quote'].apply(
    lambda x: hybrid_classify(x, keyword_dict, label_embeddings, embed_model)
)

# Step 10: Preview results
print(df[['Topic Quote', 'Keyword Labels', 'Predicted Labels']].head())


Embedding model loaded!
                                         Topic Quote  \
0  But I think just keeping in mind that not that...   
1  something that allows it to, or the people usi...   
2  I suppose, obviously designing an AI. I would ...   
3  AI, listen, I'm all for it. I'm all for artifi...   
4  But if it's there as a positive tool to help y...   

                                      Keyword Labels  \
0                                                 []   
1                                     [Adaptability]   
2                                     [Adaptability]   
3                      [Openness to Experimentation]   
4  [System Performance, Openness to Experimentati...   

                                    Predicted Labels  
0                                                 []  
1                                     [Adaptability]  
2                                     [Adaptability]  
3                      [Openness to Experimentation]  
4  [System Performance, Ope

In [135]:
# Step 11: Define the labels
labels = ['Adaptability', 'Intent to use', 'System Performance', 
          'System Understanding', 'Openness to Experimentation', 'Percieved Usefulness']

# Step 12: Convert Predicted Labels to one-hot columns
for label in labels:
    df[f'Predicted_{label}'] = df['Predicted Labels'].apply(lambda x: 1 if label in x else 0)
    df[f'Manual_{label}'] = df[label]  # assuming columns 4:10 in Excel are manual one-hot

# Step 13: Calculate summary metrics
summary_list = []

for label in labels:
    manual_count = df[f'Manual_{label}'].sum()
    predicted_count = df[f'Predicted_{label}'].sum()
    correct_matches = ((df[f'Manual_{label}'] == 1) & (df[f'Predicted_{label}'] == 1)).sum()
    
    precision = correct_matches / predicted_count if predicted_count > 0 else 0
    recall = correct_matches / manual_count if manual_count > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    summary_list.append({
        'Label': label,
        'Manual Count': manual_count,
        'Predicted Count': predicted_count,
        'Correct Matches': correct_matches,
        'Precision': round(precision, 2),
        'Recall': round(recall, 2),
        'F1 Score': round(f1, 2)
    })

# Step 14: Create summary DataFrame
summary_df = pd.DataFrame(summary_list)

# Step 15: Display summary
print(summary_df)


                         Label  Manual Count  Predicted Count  \
0                 Adaptability          75.0               58   
1                Intent to use          34.0                9   
2           System Performance          65.0               29   
3         System Understanding          83.0               33   
4  Openness to Experimentation          37.0                6   
5         Percieved Usefulness         140.0               64   

   Correct Matches  Precision  Recall  F1 Score  
0               44       0.76    0.59      0.66  
1                4       0.44    0.12      0.19  
2               21       0.72    0.32      0.45  
3               23       0.70    0.28      0.40  
4                3       0.50    0.08      0.14  
5               52       0.81    0.37      0.51  


In [136]:
# Define the labels exactly as in your DataFrame
labels = [
    'Adaptability',
    'Intent to use',
    'System Performance',
    'System Understanding',
    'Openness to Experimentation',
    'Percieved Usefulness'
]

summary_list = []
mismatch_rows = []

for label in labels:
    
    manual_col = f"Manual_{label}"
    pred_col = f"Predicted_{label}"

    # ----- Boolean masks -----
    manual_1 = df[manual_col] == 1
    pred_1 = df[pred_col] == 1

    # ----- Counts -----
    manual_count = manual_1.sum()
    predicted_count = pred_1.sum()
    correct_matches = (manual_1 & pred_1).sum()

    # ----- Metrics -----
    precision = correct_matches / predicted_count if predicted_count else 0
    recall = correct_matches / manual_count if manual_count else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    # ----- Add row to summary -----
    summary_list.append({
        'Label': label,
        'Manual Count': manual_count,
        'Predicted Count': predicted_count,
        'Correct Matches': correct_matches,
        'Precision': round(precision, 2),
        'Recall': round(recall, 2),
        'F1 Score': round(f1, 2)
    })

    # -----------------------------
    # Identify mismatches
    # -----------------------------
    false_negative_mask = manual_1 & ~pred_1       # Model missed true label
    false_positive_mask = ~manual_1 & pred_1       # Model predicted incorrectly

    # False negatives: Manual=1, Pred=0
    for idx in df[false_negative_mask].index:
        mismatch_rows.append({
            'Quote Number': df.loc[idx, 'Quote Number'],
            'Participant Number': df.loc[idx, 'Participant Number'],
            'Topic Quote': df.loc[idx, 'Topic Quote'],
            'Label Checked': label,
            'Manual Value': 1,
            'Predicted Value': 0,
            'Reason for Mismatch': 'Model missed a true label (False Negative)',
            
            # NEW: what the model DID predict
            'All Predicted Labels': [
                col.replace("Predicted_", "") 
                for col in df.columns if col.startswith("Predicted_") and df.loc[idx, col] == 1
            ],
            
            # NEW: what manual label(s) were present
            'All Manual Labels': [
                col.replace("Manual_", "") 
                for col in df.columns if col.startswith("Manual_") and df.loc[idx, col] == 1
            ]
        })

    # False positives: Manual=0, Pred=1
    for idx in df[false_positive_mask].index:
        mismatch_rows.append({
            'Quote Number': df.loc[idx, 'Quote Number'],
            'Participant Number': df.loc[idx, 'Participant Number'],
            'Topic Quote': df.loc[idx, 'Topic Quote'],
            'Label Checked': label,
            'Manual Value': 0,
            'Predicted Value': 1,
            'Reason for Mismatch': 'Model predicted wrongly (False Positive)',
            
            # NEW: what model predicted
            'All Predicted Labels': [
                col.replace("Predicted_", "") 
                for col in df.columns if col.startswith("Predicted_") and df.loc[idx, col] == 1
            ],
            
            # NEW: what manual coder applied instead
            'All Manual Labels': [
                col.replace("Manual_", "") 
                for col in df.columns if col.startswith("Manual_") and df.loc[idx, col] == 1
            ]
        })

# -----------------------------
# Convert to DataFrames
# -----------------------------
summary_df = pd.DataFrame(summary_list)
mismatch_df = pd.DataFrame(mismatch_rows)

# -----------------------------
# Export mismatches to Excel
# -----------------------------
output_file = "quote_mismatches.xlsx"
mismatch_df.to_excel(output_file, index=False)

# -----------------------------
# Print outputs
# -----------------------------
print(summary_df)
print(f"\nSaved mismatch file: {output_file}")


                         Label  Manual Count  Predicted Count  \
0                 Adaptability            75               58   
1                Intent to use            34                9   
2           System Performance            65               29   
3         System Understanding            83               33   
4  Openness to Experimentation            37                6   
5         Percieved Usefulness           140               64   

   Correct Matches  Precision  Recall  F1 Score  
0               44       0.76    0.59      0.66  
1                4       0.44    0.12      0.19  
2               21       0.72    0.32      0.45  
3               23       0.70    0.28      0.40  
4                3       0.50    0.08      0.14  
5               52       0.81    0.37      0.51  

Saved mismatch file: quote_mismatches.xlsx


In [129]:
# -----------------------------------------
# Create a dataset containing manual + predicted columns
# -----------------------------------------

output_rows = []

for idx, row in df.iterrows():
    output_row = {
        "Quote Number": row.get("Quote Number", idx),
        "Participant Number": row.get("Participant Number", None),
        "Topic Quote": row.get("Topic Quote", "")
    }
    
    # Add manual + predicted columns
    for label in labels:
        manual_col = f"Manual_{label}"
        pred_col = f"Predicted_{label}"
        
        manual_val = row.get(manual_col, 0)
        pred_val = row.get(pred_col, 0)
        
        # Manual & Predicted Values
        output_row[f"{label}_Manual"] = manual_val
        output_row[f"{label}_Predicted"] = pred_val
        
        # Match flag
        output_row[f"{label}_Match"] = 1 if manual_val == pred_val else 0
    
    output_rows.append(output_row)

# Convert to DataFrame
match_df = pd.DataFrame(output_rows)

# Save file
match_output_file = "quote_label_matches.xlsx"
match_df.to_excel(match_output_file, index=False)

print(f"Saved complete match/mismatch file: {match_output_file}")


Saved complete match/mismatch file: quote_label_matches.xlsx
